# 03 · MechDist network

Topology of the rank-mean-fused MechDist disease network and the benchmark-stripped training graph (Methods *MechDist network*; Supplementary *MechDist network degree distribution*). Reads the staged network in `data/03_network/`; the candidate-pair table (`data/zenodo/network/`) is optional.

In [1]:
from pathlib import Path
import hashlib
import numpy as np
import pandas as pd
pd.set_option("display.width", 160)
REPO = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "data" / "00_raw").is_dir())
D = REPO / "data"
RAW = D / "00_raw"
def pairs(a, b):
    return {tuple(sorted(x)) for x in zip(a, b)}
import networkx as nx

In [2]:
e = pd.read_parquet(D / "03_network/mechdist_network_edges.parquet.gzip")
nodes = pd.read_parquet(D / "03_network/mechdist_network_nodes.parquet.gzip")
M = nx.Graph(); M.add_nodes_from(nodes.disease_id); M.add_edges_from(zip(e.disease_a, e.disease_b))
deg = pd.Series(dict(M.degree()))
print("diseases", M.number_of_nodes(), "| edges", M.number_of_edges(), f"| mean degree {deg.mean():.1f}", "| components", nx.number_connected_components(M))
print("degree quantiles:", deg.quantile([0, .25, .5, .75, .99, 1]).round().astype(int).to_dict())
assert (M.number_of_nodes(), M.number_of_edges(), round(deg.mean(), 1)) == (13021, 332082, 51.0)

diseases 13021 | edges 332082 | mean degree 51.0 | components 1
degree quantiles: {0.0: 1, 0.25: 10, 0.5: 23, 0.75: 59, 0.99: 358, 1.0: 3123}


In [3]:
cc = nx.average_clustering(M)
print(f"average clustering {cc:.2f}")
assert round(cc, 2) == 0.47

average clustering 0.47


## Edges by evidence-channel class and selection rule
Edges are selected by rank within each channel class (`topN`), plus a per-disease floor of 3 pairs per class (`floor`).

In [4]:
print(pd.crosstab(e.channel_class, e.origin, margins=True))

origin         floor    topN     All
channel_class                       
all                0   12638   12638
go+hpo             0      23      23
go+px              0  100907  100907
go_only            0      33      33
hpo+px          5090  160274  165364
hpo_only       13392   39725   53117
All            18482  313600  332082


## Candidate pairs (optional, Zenodo)
Methods: *candidate disease pairs are the 5,530,050 pairs whose GO or HPO distance is at most 0.2*.

In [5]:
f = D / "zenodo/network/dis_dist_all_df_with_crank.parquet.gzip"
if f.exists():
    import pyarrow.parquet as pq
    n_cand = pq.ParquetFile(f).metadata.num_rows
    print("candidate pairs", n_cand)
    assert n_cand == 5530050
else:
    print("candidate-pair table not present (Zenodo bundle)")

candidate pairs 5530050


## Benchmark strip → training graph
Every network edge that appears in any IxIDN product (positive, negative or the matched evaluation universe) is removed before learning. Methods: *training network has 13,021 diseases and 325,837 edges*; *13,019 of the 13,021 are embedded, the remaining two are isolated*.

In [6]:
pos = pd.read_parquet(D / "02S_build_IxIDN/ixidn_pos_edges.parquet.gz")
neg = pd.read_parquet(D / "02S_build_IxIDN_neg/ixidn_neg_pinned_edges.parquet.gz")
u = pd.read_csv(RAW / "frozen/ixidn_eval_pair_universe_frozen.tsv.gz", sep="\t")
ixidn = pairs(pos.disease_1, pos.disease_2) | pairs(neg.disease_1, neg.disease_2) | pairs(u.a, u.b)
strip = [x for x in pairs(e.disease_a, e.disease_b) if x in ixidn]
T = M.copy(); T.remove_edges_from(strip)
iso = [x for x in T if T.degree(x) == 0]
print("stripped edges", len(strip), "| training edges", T.number_of_edges(), "| isolated", len(iso), "| embeddable", T.number_of_nodes() - len(iso))
assert (len(strip), T.number_of_edges(), T.number_of_nodes() - len(iso)) == (6245, 325837, 13019)

stripped edges 6245 | training edges 325837 | isolated 2 | embeddable 13019


## Benchmark diseases in the network

In [7]:
b = pd.read_csv(D / "06S_final_benchmark/ixidn_matched_4364.tsv", sep="\t")
bd = set(b.a) | set(b.b)
print("benchmark diseases in network", len(bd & set(M)), "of", len(bd))
print("benchmark pairs that are network edges:", sum(M.has_edge(x, y) for x, y in zip(b.a, b.b)), "| in training graph:", sum(T.has_edge(x, y) for x, y in zip(b.a, b.b)))
print("training-graph degree of benchmark diseases:", pd.Series([T.degree(x) for x in bd if x in T]).describe()[["min", "50%", "max"]].to_dict())

benchmark diseases in network 938 of 938
benchmark pairs that are network edges: 622 | in training graph: 0
training-graph degree of benchmark diseases: {'min': 1.0, '50%': 45.0, 'max': 2953.0}
